In [0]:
%pip install databricks-sdk

In [0]:

dbutils.widgets.text("NB_Storage_Account", "")
dbutils.widgets.text("NB_Arm_Client_Id", "")
dbutils.widgets.text("NB_Access_Connector", "")

storage_account = dbutils.widgets.get("NB_Storage_Account")
service_principal = dbutils.widgets.get("NB_Arm_Client_Id")
access_connector = dbutils.widgets.get("NB_Access_Connector")
print(storage_account)
print(service_principal)
print(access_connector)

In [0]:



from databricks.sdk import WorkspaceClient
from databricks.sdk.service.catalog import AzureManagedIdentity

w = WorkspaceClient()

try:
    w.storage_credentials.create(
        name="<credential>",
        azure_managed_identity=AzureManagedIdentity(
            access_connector_id=access_connector
        ),
        skip_validation=True
    )
    print("Done: Storage Credential created")
except Exception as e:
    if "already exists" in str(e):
        print("Storage Credential already exists, skipping")
    else:
        raise e

In [0]:
# CELL 2 - Create External Locations
spark.sql(f"CREATE EXTERNAL LOCATION IF NOT EXISTS config URL 'abfss://config@{storage_account}.dfs.core.windows.net/' WITH (STORAGE CREDENTIAL <credential>)")
spark.sql(f"CREATE EXTERNAL LOCATION IF NOT EXISTS raw URL 'abfss://raw@{storage_account}.dfs.core.windows.net/' WITH (STORAGE CREDENTIAL <credential>)")
spark.sql(f"CREATE EXTERNAL LOCATION IF NOT EXISTS bronze URL 'abfss://bronze@{storage_account}.dfs.core.windows.net/' WITH (STORAGE CREDENTIAL <credential>)")
spark.sql(f"CREATE EXTERNAL LOCATION IF NOT EXISTS silver URL 'abfss://silver@{storage_account}.dfs.core.windows.net/' WITH (STORAGE CREDENTIAL <credential>)")
spark.sql(f"CREATE EXTERNAL LOCATION IF NOT EXISTS gold URL 'abfss://gold@{storage_account}.dfs.core.windows.net/' WITH (STORAGE CREDENTIAL <credential>)")
print("Done: External Locations")

# CELL 3 - Create Catalog
spark.sql("CREATE CATALOG IF NOT EXISTS <catalog>")
print("Done: Catalog")

# CELL 4 - Create Schemas
spark.sql(f"CREATE SCHEMA IF NOT EXISTS <catalog>.bronze MANAGED LOCATION 'abfss://bronze@{storage_account}.dfs.core.windows.net/sales/'")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS <catalog>.silver MANAGED LOCATION 'abfss://silver@{storage_account}.dfs.core.windows.net/sales/'")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS <catalog>.gold MANAGED LOCATION 'abfss://gold@{storage_account}.dfs.core.windows.net/sales/'")
print("Done: Schemas")

# CELL 8 - Grant External Location Permissions
spark.sql(f"GRANT READ FILES ON EXTERNAL LOCATION config TO `{service_principal}`")
spark.sql(f"GRANT READ FILES ON EXTERNAL LOCATION raw TO `{service_principal}`")
spark.sql(f"GRANT READ FILES ON EXTERNAL LOCATION bronze TO `{service_principal}`")
spark.sql(f"GRANT WRITE FILES ON EXTERNAL LOCATION bronze TO `{service_principal}`")
spark.sql(f"GRANT READ FILES ON EXTERNAL LOCATION silver TO `{service_principal}`")
spark.sql(f"GRANT WRITE FILES ON EXTERNAL LOCATION silver TO `{service_principal}`")
spark.sql(f"GRANT READ FILES ON EXTERNAL LOCATION gold TO `{service_principal}`")
spark.sql(f"GRANT WRITE FILES ON EXTERNAL LOCATION gold TO `{service_principal}`")
print("Done: External Location Grants")

# CELL 9 - Grant Catalog Permission
spark.sql(f"GRANT USE CATALOG ON CATALOG <catalog> TO `{service_principal}`")
print("Done: Catalog Grant")

# CELL 10 - Grant Bronze Permissions
spark.sql(f"GRANT USE SCHEMA ON SCHEMA <catalog>.bronze TO `{service_principal}`")
spark.sql(f"GRANT READ METADATA ON SCHEMA <catalog>.bronze TO `{service_principal}`")
print("Done: Bronze Grants")

# CELL 11 - Grant Silver Permissions
spark.sql(f"GRANT USE SCHEMA ON SCHEMA <catalog>.silver TO `{service_principal}`")
spark.sql(f"GRANT READ METADATA ON SCHEMA <catalog>.silver TO `{service_principal}`")
print("Done: Silver Grants")

# CELL 12 - Grant Gold Permissions
spark.sql(f"GRANT USE SCHEMA ON SCHEMA <catalog>.gold TO `{service_principal}`")
spark.sql(f"GRANT READ METADATA ON SCHEMA <catalog>.gold TO `{service_principal}`")
print("Done: Gold Grants")
